### 1. Imports

In [3]:
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, TimeSeriesSplit)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

### 2. Cargar data

In [4]:
columns = ["age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"]

df = pd.read_csv(
    "../data/raw/adult.data",
    names=columns,
    skipinitialspace=True,
    na_values="?"
)

print(df.shape)
print(df.isna().sum()[df.isna().sum() > 0])

(32561, 15)
workclass         1836
occupation        1843
native_country     583
dtype: int64


### 3. Dedup por clave de negocio

In [5]:
# Clave de negocio: todas las columnas que describen al registro (sin el target).
# Si dos filas tienen la misma clave, son el mismo registro censal repetido.
# Al excluir 'income' de la clave, también eliminamos duplicados con etiqueta contradictoria.
key_cols = [c for c in df.columns if c != "income"]

n_antes = len(df)
n_dup = df.duplicated(subset=key_cols).sum()
df = df.drop_duplicates(subset=key_cols, keep="first").reset_index(drop=True)

print(f"Antes: {n_antes} | Duplicados eliminados: {n_dup} | Después: {len(df)}")

Antes: 32561 | Duplicados eliminados: 25 | Después: 32536


### 4. X e y sin leakage 

In [6]:
# Escenario: predecir si una persona gana >50K ANTES de conocer su ingreso anual.
# Pregunta por feature: ¿cuándo estaría disponible?

num_cols = [
    "age",             # Disponible siempre: dato demográfico (DNI)
    "education_num",   # Disponible al inicio: nivel educativo ya alcanzado
    "hours_per_week",  # Disponible al inicio: jornada declarada en el contrato
]
cat_cols = [
    "workclass",       # Disponible al inicio: tipo de empleador actual
    "marital_status",  # Disponible siempre: estado civil actual
    "occupation",      # Disponible al inicio: puesto actual
    "relationship",    # Disponible siempre: rol en el hogar
    "race",            # Disponible siempre: dato demográfico
    "sex",             # Disponible siempre: dato demográfico
    "native_country",  # Disponible siempre: país de origen
]

# EXCLUIDAS:
# income       -> es el target
# fnlwgt       -> peso muestral que calcula el censo DESPUÉS de la encuesta
# education    -> duplica education_num
# capital_gain -> se conoce al cierre del año, igual que el ingreso (leakage)
# capital_loss -> mismo motivo que capital_gain

X = df[num_cols + cat_cols].copy()
y = (df["income"] == ">50K").astype(int)

print("X:", X.shape, "| % positivos:", round(y.mean(), 3))

X: (32536, 10) | % positivos: 0.241


### 5. Split estratificado

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (26028, 10) | Test: (6508, 10)


### 6. Preprocesamiento y modelos en Pipeline

In [8]:
def make_prep():
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),      # necesario para LogReg y SVM; a los árboles no les afecta
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols),
    ])

models = {
    "Baseline (Dummy)": DummyClassifier(strategy="most_frequent"),
    "LogReg":           LogisticRegression(max_iter=1000),
    "Random Forest":    RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost":          XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                                      eval_metric="logloss", random_state=42, n_jobs=-1),
    "LightGBM":         LGBMClassifier(n_estimators=300, learning_rate=0.1,
                                       random_state=42, verbose=-1),
    "SVM (lineal)":     LinearSVC(C=1.0, max_iter=5000),  # lineal: SVC-RBF tarda demasiado con 26k filas
}

pipelines = {name: Pipeline([("prep", make_prep()), ("model", m)]) for name, m in models.items()}
list(pipelines)

['Baseline (Dummy)',
 'LogReg',
 'Random Forest',
 'XGBoost',
 'LightGBM',
 'SVM (lineal)']

### 7. Cross_val_score

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, pipe in pipelines.items():
    t0 = time.time()
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1")
    rows.append({"modelo": name, "f1_media": scores.mean(),
                 "f1_std": scores.std(), "tiempo_s": time.time() - t0})
    print(f"✓ {name}")

cv_results = pd.DataFrame(rows).set_index("modelo").sort_values("f1_media", ascending=False)

# Empate: media dentro de 1 desviación estándar del mejor
best = cv_results.index[0]
umbral = cv_results.loc[best, "f1_media"] - cv_results.loc[best, "f1_std"]
cv_results["estado"] = np.where(
    cv_results.index == best, "★ mejor",
    np.where(cv_results["f1_media"] >= umbral, "≈ empate con el mejor", "")
)
cv_results["F1 (media ± std)"] = cv_results.apply(
    lambda r: f"{r.f1_media:.3f} ± {r.f1_std:.3f}", axis=1)

cv_results[["F1 (media ± std)", "estado", "tiempo_s"]].round(1)

✓ Baseline (Dummy)
✓ LogReg
✓ Random Forest
✓ XGBoost
✓ LightGBM
✓ SVM (lineal)


,F1 (media ± std),estado,tiempo_s
modelo,,,
XGBoost,0.644 ± 0.009,★ mejor,2.7
LightGBM,0.639 ± 0.009,≈ empate con el mejor,2.6
LogReg,0.614 ± 0.017,,1.7
SVM (lineal),0.614 ± 0.016,,1.0
Random Forest,0.598 ± 0.006,,4.9
Baseline (Dummy),0.000 ± 0.000,,0.5


### 8. Ensamble

In [10]:
voting = Pipeline([
    ("prep", make_prep()),
    ("model", VotingClassifier(
        estimators=[
            ("lr",   LogisticRegression(max_iter=1000)),
            ("rf",   RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
            ("xgb",  XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                                   eval_metric="logloss", random_state=42, n_jobs=-1)),
            ("lgbm", LGBMClassifier(n_estimators=300, learning_rate=0.1,
                                    random_state=42, verbose=-1)),
        ],
        voting="soft"
    ))
])

t0 = time.time()
s_vot = cross_val_score(voting, X_train, y_train, cv=cv, scoring="f1")
t_vot = time.time() - t0

f1_best, std_best, t_best = cv_results.loc[best, ["f1_media", "f1_std", "tiempo_s"]]
ganancia = s_vot.mean() - f1_best

print(f"Voting:  F1 = {s_vot.mean():.3f} ± {s_vot.std():.3f} | tiempo = {t_vot:.1f}s")
print(f"{best}: F1 = {f1_best:.3f} ± {std_best:.3f} | tiempo = {t_best:.1f}s")
print(f"\nGanancia: {ganancia:+.3f} | Costo: {t_vot / t_best:.1f}x más tiempo")
print("¿Justifica su costo?",
      "SÍ: mejora más de 1 std." if ganancia > std_best
      else "NO: la mejora está dentro del ruido (≤ 1 std) y es más lento y complejo.")

Voting:  F1 = 0.638 ± 0.011 | tiempo = 10.1s
XGBoost: F1 = 0.644 ± 0.009 | tiempo = 2.7s

Ganancia: -0.006 | Costo: 3.7x más tiempo
¿Justifica su costo? NO: la mejora está dentro del ruido (≤ 1 std) y es más lento y complejo.


### 9. Validación temporal vs aleatorio

In [11]:
# Adult no tiene columna de año/fecha. Usamos el ORDEN DEL ARCHIVO como pseudo-tiempo:
# TimeSeriesSplit entrena siempre con filas "anteriores" y valida con filas "posteriores".
X_tr_ord = X_train.sort_index()
y_tr_ord = y_train.sort_index()

tscv = TimeSeriesSplit(n_splits=5)
pipe_best = pipelines[best]

s_temp = cross_val_score(pipe_best, X_tr_ord, y_tr_ord, cv=tscv, scoring="f1")
s_rand = cross_val_score(pipe_best, X_train, y_train, cv=cv, scoring="f1")

comp = pd.DataFrame({
    "split": ["Aleatorio (StratifiedKFold)", "Temporal (TimeSeriesSplit)"],
    "F1 media": [s_rand.mean(), s_temp.mean()],
    "F1 std": [s_rand.std(), s_temp.std()],
}).round(3)

print(f"Modelo: {best}")
print("Folds temporales:", np.round(s_temp, 3))
print(f"Diferencia (aleatorio - temporal): {s_rand.mean() - s_temp.mean():+.3f}")
comp

Modelo: XGBoost
Folds temporales: [0.595 0.642 0.622 0.64  0.659]
Diferencia (aleatorio - temporal): +0.012


,split,F1 media,F1 std
0,Aleatorio (StratifiedKFold),0.644,0.009
1,Temporal (TimeSeriesSplit),0.632,0.022


### 10. Evaluación final en test

In [12]:
final_model = pipelines[best].fit(X_train, y_train)
f1_test = f1_score(y_test, final_model.predict(X_test))
print(f"{best} -> F1 test = {f1_test:.3f} | F1 CV = {f1_best:.3f}")

XGBoost -> F1 test = 0.657 | F1 CV = 0.644


### 11. Párrafo final

## ¿Qué modelo llevaría a la Semana 4?

Llevaría **XGBoost**, porque obtuvo el mejor F1 en validación cruzada (**0.644 ± 0.009**) y en test alcanzó **0.657**. Ese valor es muy cercano al de CV, así que generaliza bien y no está sobreajustado. **LightGBM** quedó empatado con él (0.639 ± 0.009): la diferencia de 0.005 es menor a una desviación estándar y tardan prácticamente lo mismo (2.6 s vs 2.7 s), así que sería una alternativa igual de válida. Elijo XGBoost por tener la media más alta y por haberlo confirmado en test. LogReg y SVM (≈ 0.614) quedan claramente por debajo, y Random Forest (0.598) es el peor de los modelos reales, además del más lento. El ensamble por soft voting **no justifica su costo**: obtuvo 0.638 ± 0.011, es decir 0.006 *menos* que XGBoost solo, y tarda 3.7 veces más. En la validación temporal, XGBoost obtuvo 0.632 ± 0.022 frente a 0.644 ± 0.009 con el split aleatorio. La caída de 0.012 está dentro de la desviación del split temporal, pero los folds fueron más variables (de 0.595 a 0.659), así que el split aleatorio es algo optimista respecto de la estabilidad. Como Adult no tiene fechas reales, esto solo indica que no hay un drift fuerte a lo largo del archivo; con datos fechados por año habría que repetir la prueba antes de llevar el modelo a producción.